In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
_r = pathlib.Path.cwd().resolve()
while not (_r / "config" / "settings.yaml").exists() and _r != _r.parent:
    _r = _r.parent
sys.path.insert(0, str(_r))                      # makes `import src...` work from any folder
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
from src.notebook_utils import *
sns.set_theme(style="whitegrid"); pd.set_option("display.max_columns", 40, "display.width", 140)
print("project root:", _r)

project root: /app


# 07 - Model evaluation
Reloads the **serialised** models and evaluates on the untouched test partition: accuracy, precision, recall, F1, ROC-AUC, PR-AUC, confusion matrices, false-positive rate, threshold trade-offs, calibration and risk tiers.

In [2]:
from src.model_evaluation.evaluate import evaluate_binary, comparison_table, bootstrap_ci
from src.fraud_detection.risk import tier_summary
summ = load_summary(); fe = load_features(); d = split_xy(fe); X_te, y_te, te = d['test']
names = ['logistic_regression', 'random_forest', 'xgboost']
models = {n: load_model(n) for n in names}; scores = {n: m.predict_proba(X_te)[:, 1] for n, m in models.items()}
res = {n: evaluate_binary(y_te, scores[n], summ['models']['results'][n]['threshold']) for n in names}
comparison_table(res).round(4)

,accuracy,precision,recall,f1,roc_auc,pr_auc,false_positive_rate,tp,fp,fn,tn
xgboost,0.9925,0.9820,0.6264,0.7649,0.8854,0.7429,0.0002,109.0,2.0,65.0,8777.0
random_forest,0.9925,0.9280,0.6667,0.7759,0.9026,0.7412,0.0010,116.0,9.0,58.0,8770.0
logistic_regression,0.9901,0.8829,0.5632,0.6877,0.8722,0.6658,0.0015,98.0,13.0,76.0,8766.0


In [3]:
ci = {n: bootstrap_ci(y_te, scores[n], n_boot=200) for n in names}; pd.DataFrame(ci, index=['ROC-AUC 2.5%', 'ROC-AUC 97.5%']).T.round(4)

,ROC-AUC 2.5%,ROC-AUC 97.5%
logistic_regression,0.8337,0.9150
random_forest,0.8686,0.9366
xgboost,0.8441,0.9246


In [4]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, n in zip(ax, names):
    r = res[n]; sns.heatmap([[r['tn'], r['fp']], [r['fn'], r['tp']]], annot=True, fmt='d', cmap='Blues', cbar=False, ax=a, xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud']); a.set_title(f"{n}\nFPR {r['false_positive_rate']:.3%}")
plt.tight_layout()

In [5]:
# threshold trade-off for the champion
ch = summ['models']['champion']; rows = [dict(threshold=t, **{k: v for k, v in evaluate_binary(y_te, scores[ch], t).items() if k in ('precision', 'recall', 'f1', 'false_positive_rate')}) for t in np.linspace(0.02, 0.98, 25)]
pd.DataFrame(rows).set_index('threshold')[['precision', 'recall', 'f1', 'false_positive_rate']].plot(figsize=(8, 4), title=f'Threshold sweep - {ch}')

<Axes: title={'center': 'Threshold sweep - xgboost'}, xlabel='threshold'>

In [6]:
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(5, 5))
for n in names:
    fp_, mp_ = calibration_curve(y_te, scores[n], n_bins=8, strategy='quantile'); ax.plot(mp_, fp_, marker='o', label=n)
ax.plot([0, 1], [0, 1], 'k--'); ax.set(xlabel='mean predicted probability', ylabel='observed fraud rate', title='Calibration'); ax.legend()

In [7]:
tier_summary(scores[ch], y_te, te.amount.to_numpy(), CFG['risk_tiers']).round(4)

,tier,transactions,frauds,amount,observed_fraud_rate,action
0,low,3,0,514.20,0.0000,allow
1,medium,7871,37,533729.42,0.0047,step-up authentication
2,high,912,12,91560.36,0.0132,hold for manual review
3,critical,167,125,34054.96,0.7485,block + alert


In [8]:
# business view: money saved vs friction (flagged legit transactions)
thr = summ['models']['results'][ch]['threshold']; pred = scores[ch] >= thr; amt = te.amount.to_numpy()
print(f'fraud value caught: {amt[pred & (y_te==1)].sum():,.0f} | missed: {amt[~pred & (y_te==1)].sum():,.0f} | legit customers flagged: {int((pred & (y_te==0)).sum())} of {int((y_te==0).sum())}')

fraud value caught: 24,794 | missed: 4,948 | legit customers flagged: 2 of 8779


**Reading the false-positive rate:** even FPR ≈ 0.5% means thousands of wrongly challenged customers at scale - a direct link between model quality and consumer trust.